# Шаг 3. Обучаем голос (Piper) и выгружаем готовый файл

Этот блокнот берёт записи из шага 2, **дообучает** на них маленькую офлайн-модель Piper
и выдаёт два файла, которые загружаются в Агент.

**Сначала выберите видеокарту:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU** → «Сохранить».
Нажимайте ▶ **по порядку, сверху вниз**.

> ⚠️ **Про лицензию.** Дообучение идёт от готового русского голоса Piper (`denis`). Его записи под лицензией CC0,
> но сам он в свою очередь дообучен от английского голоса `lessac`, чьи записи выпущены **только для исследований**
> (Blizzard 2013). Юридически, можно ли встраивать такой голос в платный продукт — **не ясно**. Для личного
> использования это не проблема. Для продажи читайте `docs/VOICE_STUDIO.md`, раздел «Лицензия».

## 1. Подключаем Google Диск

In [ ]:
#@title Подключить Диск и проверить видеокарту { display-mode: "form" }
import glob
import os
import subprocess
import sys

from IPython.display import Audio, display


def run(*command, cwd=None):
    """Запустить команду и показывать её вывод по мере появления. Ошибка команды останавливает блокнот."""
    process = subprocess.Popen([str(part) for part in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, cwd=cwd)
    for line in process.stdout:
        print(line, end='')
    if process.wait():
        raise RuntimeError('Команда завершилась с ошибкой (строки выше подскажут, с какой). Если не поймёте — '
                           'скопируйте последние 20 строк и покажите их в поддержке.')

from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/andrey_voice'  #@param {type:"string"}
os.makedirs(WORK, exist_ok=True)
os.makedirs('/content/voice_tools', exist_ok=True)

gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print('Видеокарта:', gpu or 'НЕ НАЙДЕНА — меню «Среда выполнения» → «Сменить среду выполнения» → T4 GPU → «Сохранить».')

In [ ]:
#@title Проверить записи { display-mode: "form" }
if os.path.exists(f'{WORK}/index.tsv') and glob.glob(f'{WORK}/wavs/*.wav'):
    print('Записи найдены:', len(glob.glob(f'{WORK}/wavs/*.wav')), 'файлов.')
else:
    print('Записей нет. Сначала пройдите блокнот andrey_voice_1_zapis.ipynb (шаг 2).')

## 2. Загружаем программы подготовки

In [ ]:
%%writefile /content/voice_tools/make_dataset.py
"""Шаг 3. Подготовить набор для обучения Piper из записей Qwen3-TTS.

  python make_dataset.py --src /content/drive/MyDrive/andrey_voice --out /content/drive/MyDrive/andrey_voice/dataset

Что делает с каждой записью:
  * приводит к 22050 Гц, моно, 16 бит — формат, с которым обучается Piper «medium»;
  * обрезает тишину в начале и в конце (оставляет короткую паузу);
  * выравнивает громкость;
  * отбрасывает брак: слишком короткие и длинные, тихие, с перегрузом (хрип).
Результат: папка wavs/ и файл metadata.csv в формате «имя.wav|текст», который читает piper.train.
"""
import argparse
import math
import wave
from pathlib import Path

import numpy as np

TARGET_RATE = 22050
MIN_SECONDS, MAX_SECONDS = 0.8, 14.0
SILENCE_DB = -45.0          # тише этого — считаем паузой
KEEP_PAD = 0.15             # сколько секунд тишины оставить по краям
PEAK_DB = -1.5              # громкость после выравнивания (пик)


def read_wav(path):
    """(звук float32 моно, частота). Читает WAV 16 бит стандартным модулем, остальное — через soundfile."""
    try:
        with wave.open(str(path), 'rb') as file:
            rate, channels, width = file.getframerate(), file.getnchannels(), file.getsampwidth()
            raw = file.readframes(file.getnframes())
        if width != 2:
            raise ValueError('не 16 бит')
        data = np.frombuffer(raw, dtype='<i2').astype(np.float32) / 32768.0
        if channels > 1:
            data = data.reshape(-1, channels).mean(axis=1)
        return data, rate
    except (wave.Error, ValueError, EOFError):
        import soundfile
        data, rate = soundfile.read(str(path), dtype='float32')
        if data.ndim > 1:
            data = data.mean(axis=1)
        return data.astype(np.float32), rate


def write_wav(path, data, rate):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    pcm = np.clip(data, -1.0, 1.0)
    with wave.open(str(path), 'wb') as file:
        file.setnchannels(1)
        file.setsampwidth(2)
        file.setframerate(rate)
        file.writeframes((pcm * 32767.0).astype('<i2').tobytes())


def resample(data, rate_from, rate_to=TARGET_RATE):
    """Привести частоту. В Colab есть SciPy (качественный полифазный фильтр); на ПК без SciPy — запасной путь:
    сглаживание от наложения частот и линейная интерполяция (для проверки и небольших наборов достаточно)."""
    if rate_from == rate_to:
        return data
    try:
        from scipy.signal import resample_poly
    except ImportError:
        return _resample_numpy(data, rate_from, rate_to)
    divisor = math.gcd(int(rate_from), int(rate_to))
    return resample_poly(data, int(rate_to) // divisor, int(rate_from) // divisor).astype(np.float32)


def _resample_numpy(data, rate_from, rate_to):
    data = np.asarray(data, dtype=np.float32)
    if not len(data):
        return data
    if rate_to < rate_from:
        # Простой фильтр нижних частот (окно Хэннинга) перед понижением частоты — без «звона» от наложения.
        cutoff = rate_to / rate_from / 2
        taps = 63
        n = np.arange(taps) - (taps - 1) / 2
        kernel = 2 * cutoff * np.sinc(2 * cutoff * n) * np.hanning(taps)
        data = np.convolve(data, (kernel / kernel.sum()).astype(np.float32), mode='same')
    length = int(round(len(data) * rate_to / rate_from))
    positions = np.linspace(0, len(data) - 1, num=max(1, length))
    return np.interp(positions, np.arange(len(data)), data).astype(np.float32)


def trim_silence(data, rate, threshold_db=SILENCE_DB, pad=KEEP_PAD):
    """Убрать тишину по краям, оставив небольшую паузу."""
    if not len(data):
        return data
    threshold = 10 ** (threshold_db / 20)
    loud = np.flatnonzero(np.abs(data) > threshold)
    if not len(loud):
        return data[:0]
    margin = int(pad * rate)
    return data[max(0, loud[0] - margin): loud[-1] + margin + 1]


def normalize_peak(data, peak_db=PEAK_DB):
    peak = float(np.max(np.abs(data))) if len(data) else 0.0
    return data if peak < 1e-6 else data * (10 ** (peak_db / 20) / peak)


def problem(data, rate, original_peak=None):
    """Причина отбраковки или None."""
    if not len(data):
        return 'почти тишина'                              # после обрезки тишины ничего не осталось
    seconds = len(data) / rate if rate else 0
    if seconds < MIN_SECONDS:
        return 'слишком короткая'
    if seconds > MAX_SECONDS:
        return 'слишком длинная'
    rms = float(np.sqrt(np.mean(np.square(data)))) if len(data) else 0.0
    if rms < 10 ** (-50 / 20):
        return 'почти тишина'
    if original_peak is not None and original_peak >= 0.999:
        clipped = float(np.mean(np.abs(data) >= 0.999 * original_peak))
        if clipped > 0.002:
            return 'перегруз (хрип)'
    return None


def read_index(src):
    """Таблица «файл → текст», которую пишет qwen_generate.py clone."""
    rows = []
    with open(Path(src) / 'index.tsv', encoding='utf-8') as file:
        for line in file:
            name, _, text = line.rstrip('\n').partition('\t')
            if name and text:
                rows.append((name, text))
    return rows


def build(src, out, rate=TARGET_RATE, log=print):
    src, out = Path(src), Path(out)
    wav_dir = out / 'wavs'
    wav_dir.mkdir(parents=True, exist_ok=True)
    rows, kept, dropped, total = [], [], {}, 0.0
    for name, text in read_index(src):
        source = src / 'wavs' / name
        if not source.exists():
            dropped['нет файла'] = dropped.get('нет файла', 0) + 1
            continue
        data, original_rate = read_wav(source)
        original_peak = float(np.max(np.abs(data))) if len(data) else 0.0
        data = trim_silence(resample(data, original_rate, rate), rate)
        reason = problem(data, rate, original_peak)
        if reason:
            dropped[reason] = dropped.get(reason, 0) + 1
            continue
        write_wav(wav_dir / name, normalize_peak(data), rate)
        kept.append((name, text.replace('|', ',')))       # «|» — разделитель в таблице
        total += len(data) / rate
    with open(out / 'metadata.csv', 'w', encoding='utf-8', newline='') as file:
        for name, text in kept:                            # «имя.wav|текст» — без кавычек и экранирования
            file.write(f'{name}|{" ".join(text.split())}\n')
    log(f'Годных записей: {len(kept)} ({total / 60:.0f} мин). Отброшено: '
        + (', '.join(f'{k} — {v}' for k, v in dropped.items()) or 'ничего'))
    if total < 40 * 60:
        log('Меньше 40 минут: голос получится, но нестабильным. Добавьте фраз (make_phrases.py --corpus) и повторите шаг 2.')
    log(f'Папка для обучения: {out}  (wavs/ и metadata.csv)')
    return {'kept': len(kept), 'minutes': total / 60, 'dropped': dropped}


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--src', required=True, help='папка с результатом шага 2 (внутри wavs/ и index.tsv)')
    parser.add_argument('--out', required=True, help='куда сложить готовый набор')
    args = parser.parse_args(argv)
    build(args.src, args.out)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/voice_tools/finish_voice.py
"""Шаг 5. Подготовить обученный голос к загрузке в приложение.

  python finish_voice.py --onnx model.onnx --config config.json --name andrey --title "Дворецкий" --out готовый_голос

Берёт два файла, которые у вас получились после обучения:
  model.onnx   — результат экспорта (шаг 4);
  config.json  — файл, записанный во время обучения (параметр --data.config_path).
Проверяет их, даёт правильные имена и название голоса. В папке --out появятся «ru_RU-<имя>-medium.onnx» и
«ru_RU-<имя>-medium.onnx.json»: их и загружают в Агент (Настройки → «Озвучка» → «Свой голос» → «Загрузить»).
"""
import argparse
import json
import re
import shutil
import sys
from pathlib import Path


class VoiceError(ValueError):
    pass


def check_config(data):
    """Те же признаки, по которым приложение узнаёт голос Piper. Возвращает список замечаний (не ошибок)."""
    if not isinstance(data, dict) or ('phoneme_id_map' not in data and 'audio' not in data):
        raise VoiceError('Это не файл настроек голоса Piper. Нужен config.json, который записало обучение '
                         '(параметр --data.config_path).')
    notes = []
    rate = (data.get('audio') or {}).get('sample_rate')
    if rate != 22050:
        notes.append(f'Частота {rate} Гц, а у стандартных голосов 22050 Гц — убедитесь, что обучали с --model.sample_rate 22050.')
    voice = (data.get('espeak') or {}).get('voice')
    if data.get('phoneme_type', 'espeak') == 'espeak' and voice != 'ru':
        notes.append(f'Язык произношения «{voice}», а для русского голоса нужен «ru» (--data.espeak_voice ru).')
    if int(data.get('num_speakers', 1) or 1) != 1:
        notes.append('В голосе несколько дикторов — приложение будет использовать первого.')
    return notes


def slug(name):
    value = re.sub(r'[^A-Za-z0-9_]+', '_', str(name)).strip('_').lower()
    if not value:
        raise VoiceError('Имя голоса — латинскими буквами или цифрами, например: andrey')
    return value


def finish(onnx, config, name, title, out):
    onnx, config, out = Path(onnx), Path(config), Path(out)
    if not onnx.is_file() or onnx.stat().st_size < 1_000_000:
        raise VoiceError(f'Файл {onnx} не найден или слишком мал для голоса (меньше 1 МБ). Проверьте экспорт (шаг 4).')
    try:
        data = json.loads(config.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        raise VoiceError(f'Файл {config} не читается как JSON.')
    notes = check_config(data)
    stem = f'ru_RU-{slug(name)}-medium'
    data['title'] = title or name
    data.setdefault('language', {'code': 'ru_RU', 'family': 'ru', 'region': 'RU', 'name_native': 'Русский',
                                 'name_english': 'Russian', 'country_english': 'Russia'})
    data['dataset'] = slug(name)
    out.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(onnx, out / f'{stem}.onnx')
    (out / f'{stem}.onnx.json').write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    return stem, notes


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--onnx', required=True)
    parser.add_argument('--config', required=True)
    parser.add_argument('--name', required=True, help='короткое имя латиницей, например andrey')
    parser.add_argument('--title', default='', help='как голос будет называться в приложении')
    parser.add_argument('--out', default='готовый_голос')
    args = parser.parse_args(argv)
    try:
        stem, notes = finish(args.onnx, args.config, args.name, args.title, args.out)
    except VoiceError as error:
        sys.exit(f'Ошибка: {error}')
    for note in notes:
        print('Внимание:', note)
    print(f'Готово: {args.out}/{stem}.onnx и {stem}.onnx.json')
    print('Дальше: Агент → Настройки → «Озвучка» → «Добавить голос» → «Свой голос» → выберите оба файла → «Загрузить».')


if __name__ == '__main__':
    main()

## 3. Готовим набор для обучения

Приводит звук к нужному формату, убирает тишину по краям, выравнивает громкость и выбрасывает брак.
Ячейка напишет, сколько записей годится. **Меньше 40 минут — голос будет нестабильным.**

In [ ]:
#@title Подготовить набор { display-mode: "form" }
DATASET = f'{WORK}/dataset'
run(sys.executable, '/content/voice_tools/make_dataset.py', '--src', WORK, '--out', DATASET)

## 4. Устанавливаем обучалку Piper

Занимает 5–10 минут. В конце должно появиться слово «Готово». Если ячейка упала — перезапустите среду
(«Среда выполнения» → «Перезапустить сеанс»), пройдите шаги 1–3 заново и запустите её снова.

In [ ]:
#@title Установить Piper для обучения { display-mode: "form" }
PIPER = '/content/piper1-gpl'
if not os.path.isdir(PIPER):
    run('apt-get', 'install', '-y', '-qq', 'build-essential', 'cmake', 'ninja-build', 'espeak-ng')
    run('git', 'clone', '-q', 'https://github.com/OHF-Voice/piper1-gpl.git', PIPER)
    run('git', 'checkout', '-q', 'efffbfb226bfb511ebbcf55d0cecd8b35a89743d', cwd=PIPER)
    run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train]', cwd=PIPER)
    run('bash', './build_monotonic_align.sh', cwd=PIPER)
    run(sys.executable, 'setup.py', 'build_ext', '--inplace', cwd=PIPER)
print('Готово.')

## 5. Берём готовый голос за основу

Дообучение от готового голоса в разы быстрее и надёжнее, чем с нуля (на ~40 минутах речи с нуля голос почти
наверняка не получится). Файл большой (~850 МБ), скачивается один раз.

In [ ]:
#@title Скачать основу (denis) { display-mode: "form" }
BASE_CKPT = f'{WORK}/base/denis_medium.ckpt'
os.makedirs(os.path.dirname(BASE_CKPT), exist_ok=True)
if not os.path.exists(BASE_CKPT) or os.path.getsize(BASE_CKPT) < 100_000_000:
    run('wget', '-q', '--show-progress', '-O', BASE_CKPT, 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/denis/medium/epoch%3D4474-step%3D1521860.ckpt')
print('Основа:', BASE_CKPT, round(os.path.getsize(BASE_CKPT) / 1e6), 'МБ')

## 6. Обучаем

* `NAME` — короткое имя латиницей (оно же станет именем файла).
* `EPOCHS` — сколько проходов по записям. Больше — лучше, но дольше; 200 — разумное начало.
* `BATCH` — сколько фраз за раз. Если пишет «CUDA out of memory» — поставьте 8, потом 4.

Обучение идёт **часами**. Не закрывайте вкладку. Текст в ячейке будет бежать — это нормально.
Контрольные точки сохраняются на Google Диск каждую эпоху, поэтому обрыв не страшен:
запустите ячейку «Продолжить обучение» ниже. Если Colab при этом сбросил среду, сначала заново запустите
ячейки 1 (Диск), 2 (программы) и 4 (установка Piper); шаги 3 и 5 повторять не нужно — результат лежит на Диске.

**Послушать промежуточный результат** можно уже после 30–50 эпох: нажмите ■ (остановить), пройдите шаг 7,
затем «Продолжить обучение».

In [ ]:
#@title Начать обучение { display-mode: "form" }
NAME = "andrey"  #@param {type:"string"}
EPOCHS = 200  #@param {type:"integer"}
BATCH = 16  #@param {type:"integer"}

os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # наши контрольные точки — свои, им можно доверять
run(sys.executable, '-m', 'piper.train', 'fit',
    '--data.voice_name', NAME,
    '--data.csv_path', f'{DATASET}/metadata.csv',
    '--data.audio_dir', f'{DATASET}/wavs',
    '--model.sample_rate', 22050,
    '--data.espeak_voice', 'ru',
    '--data.cache_dir', f'{WORK}/cache',
    '--data.config_path', f'{WORK}/config.json',
    '--data.batch_size', BATCH,
    '--trainer.max_epochs', EPOCHS,
    '--trainer.default_root_dir', f'{WORK}/training',
    '--model.mos_metric', 'none',
    '--model.warmstart_ckpt', BASE_CKPT,
    cwd=PIPER)

In [ ]:
#@title Продолжить обучение (после обрыва или паузы) { display-mode: "form" }
NAME = "andrey"  #@param {type:"string"}
EPOCHS = 200  #@param {type:"integer"}
BATCH = 16  #@param {type:"integer"}

DATASET = f'{WORK}/dataset'
lasts = sorted(glob.glob(f'{WORK}/training/**/last.ckpt', recursive=True), key=os.path.getmtime)
if not lasts:
    raise SystemExit('Контрольных точек нет — обучение ещё не начиналось. Запустите ячейку «Начать обучение».')
print('Продолжаю с', lasts[-1])
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
run(sys.executable, '-m', 'piper.train', 'fit',
    '--data.voice_name', NAME,
    '--data.csv_path', f'{DATASET}/metadata.csv',
    '--data.audio_dir', f'{DATASET}/wavs',
    '--model.sample_rate', 22050,
    '--data.espeak_voice', 'ru',
    '--data.cache_dir', f'{WORK}/cache',
    '--data.config_path', f'{WORK}/config.json',
    '--data.batch_size', BATCH,
    '--trainer.max_epochs', EPOCHS,
    '--trainer.default_root_dir', f'{WORK}/training',
    '--model.mos_metric', 'none',
    '--ckpt_path', lasts[-1],
    cwd=PIPER)

## 7. Слушаем, что получилось

Берёт самую свежую контрольную точку, превращает её в рабочий файл голоса и произносит три фразы.
Запускайте, когда обучение закончилось или вы его остановили (■). Два голоса сразу Colab не обучает и не озвучивает.

Что слушать: чёткие слова, ровный темп, нет «жестяного» дребезга и странных ударений.
Мелкие огрехи в ударениях бывают — их можно поправить, добавив в `phrases.txt` проблемные слова и записав заново.

In [ ]:
#@title Экспорт и прослушивание { display-mode: "form" }
TEST_SENTENCES = [
    "Добрый вечер. Все системы работают в штатном режиме.",
    "Открываю браузер и ищу информацию, которую вы просили.",
    "Напоминаю: через десять минут начинается встреча.",
]  #@param {type:"raw"}

checkpoints = sorted(glob.glob(f'{WORK}/training/**/*.ckpt', recursive=True), key=os.path.getmtime)
if not checkpoints:
    raise SystemExit('Контрольных точек нет — сначала обучение.')
chosen = [c for c in checkpoints if c.endswith('last.ckpt')][-1:] or checkpoints[-1:]
print('Беру:', chosen[0])
test = f'{WORK}/test'
os.makedirs(test, exist_ok=True)
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
run(sys.executable, '-m', 'piper.train.export_onnx', '--checkpoint', chosen[0], '--output-file', f'{test}/model.onnx', cwd=PIPER)
import shutil
shutil.copyfile(f'{WORK}/config.json', f'{test}/model.onnx.json')
for number, sentence in enumerate(TEST_SENTENCES, 1):
    run(sys.executable, '-m', 'piper', '-m', f'{test}/model.onnx', '-c', f'{test}/model.onnx.json',
        '-f', f'{test}/test_{number}.wav', '--', sentence)
    print(sentence)
    display(Audio(f'{test}/test_{number}.wav'))

## 8. Выгружаем готовый голос

Когда результат устраивает. Ячейка проверит файлы, даст им правильные имена и положит их в папку
`andrey_voice/готовый_голос` на вашем Google Диске. Оттуда скачайте оба файла на компьютер
(правая кнопка → «Скачать») и загрузите в Агент.

In [ ]:
#@title Собрать готовый голос { display-mode: "form" }
NAME = "andrey"  #@param {type:"string"}
TITLE = "Дворецкий"  #@param {type:"string"}

run(sys.executable, '/content/voice_tools/finish_voice.py', '--onnx', f'{WORK}/test/model.onnx', '--config', f'{WORK}/config.json',
    '--name', NAME, '--title', TITLE, '--out', f'{WORK}/готовый_голос')